In [ ]:
import os
import librosa
import numpy as np
import cv2
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import matplotlib
matplotlib.use('Agg') 
import matplotlib.pyplot as plt
import seaborn as sns


BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, "data", "gtzan") # Relative to project root
FEAT_DIR = os.path.join(BASE_DIR, "features")
# SETTINGS
GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop', 
          'jazz', 'metal', 'pop', 'reggae', 'rock']
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

I0000 00:00:1778830782.215785    8726 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1778830782.307994    8726 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1778830784.749311    8726 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
import os
import librosa
import cv2
import numpy as np
from tqdm import tqdm
import tensorflow as tf

def get_spectrogram(path):
    try:
        y, sr = librosa.load(path, duration=30.0)
        S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
        S_db = librosa.power_to_db(S, ref=np.max)
        S_db_norm = cv2.normalize(S_db, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
        img_resized = cv2.resize(S_db_norm, (224, 224))
        return np.stack([img_resized] * 3, axis=-1)
    except Exception as e:
        return None

X, y = [], []

for label, genre in enumerate(GENRES):
    folder = os.path.join(DATA_DIR, genre)
    if not os.path.exists(folder): continue
    
    files = [f for f in os.listdir(folder) if f.endswith('.wav') or f.endswith('.au')]
    
    for file in tqdm(files, desc=f"Processing {genre}"):
        path = os.path.join(folder, file)
        spec = get_spectrogram(path)
        if spec is not None:
            X.append(spec)
            y.append(label)

X = np.array(X, dtype='float32') / 255.0
y = tf.keras.utils.to_categorical(np.array(y), num_classes=10)

print(f"✅ Final Data Shape: {X.shape}")
print(f"✅ Final Label Shape: {y.shape}")

Processing blues:   0%|          | 0/100 [00:00<?, ?it/s]/home/ulas/Uni/Neuroinformatik/Music_Classification_Project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Processing jazz:  12%|█▏        | 12/100 [00:00<00:04, 17.70it/s]/tmp/ipykernel_8726/694248113.py:10: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(path, duration=30.0)
/home/ulas/Uni/Neuroinformatik/Music_Classification_Project/.venv/lib/python3.12/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
Processing rock: 100%|██████████| 100/100 [00:06<00:00, 16.45it/s]


✅ Final Data Shape: (999, 224, 224, 3)
✅ Final Label Shape: (999, 10)


In [3]:
# --- REPLACING CELLS 3 & 4 ---
X_list, y_list = [], []

for label, genre in enumerate(GENRES):
    folder = os.path.join(DATA_DIR, genre)
    if not os.path.exists(folder): continue
    files = [f for f in os.listdir(folder) if f.endswith('.wav') or f.endswith('.au')]
    
    for file in tqdm(files, desc=f"Processing {genre}"):
        path = os.path.join(folder, file)
        spec = get_spectrogram(path)
        if spec is not None:
            X_list.append(spec)
            y_list.append(label)

# FILTER & CONVERT: This fixes the Inhomogeneous Shape error
X = np.array(X_list, dtype='float32') / 255.0
y = np.array(y_list)

# Categorical encoding (Ensure it's only done ONCE)
y_cat = tf.keras.utils.to_categorical(y, num_classes=10)

# Stratified Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_cat, test_size=0.2, stratify=y, random_state=42
)

# Dataset creation (Reduced batch size to help with the Memory Allocation error)
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_ds = train_ds.shuffle(len(X_train)).batch(16).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(16).prefetch(tf.data.AUTOTUNE)

print(f"✅ Data Ready: {X.shape} | Labels: {y_cat.shape}")

Processing jazz:  12%|█▏        | 12/100 [00:00<00:04, 18.41it/s]/tmp/ipykernel_8726/694248113.py:10: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(path, duration=30.0)
/home/ulas/Uni/Neuroinformatik/Music_Classification_Project/.venv/lib/python3.12/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
Processing rock: 100%|██████████| 100/100 [00:05<00:00, 17.92it/s]


✅ Data Ready: (999, 224, 224, 3) | Labels: (999, 10)


E0000 00:00:1778830901.226601    8726 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
W0000 00:00:1778830901.228216    8726 cpu_allocator_impl.cc:82] Allocation of 481087488 exceeds 10% of free system memory.


In [4]:
import tensorflow as tf

# 1. Load the ResNet50V2 base
BaseModel = tf.keras.applications.ResNet50V2

base_model = BaseModel(
    weights='imagenet', 
    include_top=False, 
    input_shape=(224, 224, 3)
)

# 2. Freeze the first 100 layers (Transfer Learning)
base_model.trainable = True
for layer in base_model.layers[:100]:
    layer.trainable = False

# 3. Add Custom Head for Music Genres
inputs = tf.keras.Input(shape=(224, 224, 3))
x = tf.keras.applications.resnet_v2.preprocess_input(inputs) 
x = base_model(x)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(256, activation='relu')(x)
x = tf.keras.layers.Dropout(0.5)(x)
outputs = tf.keras.layers.Dense(10, activation='softmax')(x)

# 4. Initialize the Model
model = tf.keras.Model(inputs, outputs)

# 5. Compile
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("✅ Model defined and compiled. 'model' is now recognized.")

✅ Model defined and compiled. 'model' is now recognized.


In [5]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2)
    ]
)

Epoch 1/20


W0000 00:00:1778830903.496807    8726 cpu_allocator_impl.cc:82] Allocation of 481087488 exceeds 10% of free system memory.


 2/50 ━━━━━━━━━━━━━━━━━━━━ 2:16 3s/step - accuracy: 0.2031 - loss: 2.5293  

KeyboardInterrupt: 

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Added EarlyStopping to prevent overfitting
history_fine = model.fit(
    train_ds, 
    validation_data=val_ds, 
    epochs=20,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=5,
            restore_best_weights=True
        )
    ]
)

In [ ]:
model.save('texture_specialist_v1.h5')
texture_probs = model.predict(val_ds)
np.save('texture_probs_test.npy', texture_probs)
print("Handshake Complete. Texture Specialist ready for Ensemble.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# 1. Silent Predictions
y_pred_probs = model.predict(val_ds, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

# 2. Get true labels correctly from batched dataset
y_true = np.concatenate([y for x, y in val_ds], axis=0)
y_true = np.argmax(y_true, axis=1)

# 3. Save Confusion Matrix (No printing)
cm = confusion_matrix(y_true, y_pred, normalize='true')
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues', 
            xticklabels=GENRES, yticklabels=GENRES)
plt.savefig('resnet_confusion_matrix.png')
plt.close()

# 4. Save Model for later use in the Ensemble
model.save('models/ventral_resnet_specialist.keras')